# Inspector BIM — Edificação de Apoio Agrícola

**Master em IA para Arquitectura, Engenharia e Construção — Zigurat Institute of Technology**

**Módulo 5 — Tema 2: Agentic IA | Tarefa Individual**

**Daniel Veludo**

---

## Objectivo

Sistema multi-agente em **LangGraph** que lê um modelo IFC de uma edificação agrícola (*Casa de Máquinas — Quinta do Panascal*) e produz automaticamente:

- Verificação de conformidade (ventilação, acessibilidade, armazenamento de fitofármacos)
- Quantitativos de elementos construtivos
- Recomendações técnicas geradas por LLM (condicionais)
- Três entregáveis: **relatório `.docx`**, **checklist `.xlsx`**, **log `.json`**

## Arquitectura do Sistema

```
START
  ↓
[Agente 1: Extrator IFC]     ← IfcOpenShell lê o ficheiro .ifc
  ↓
[Agente 2: Verificador]      ← verifica ventilação, acessibilidade, fitofármacos
  ↓
[Agente 3: Quantificador]    ← calcula m² de paredes, janelas, portas, lajes
  ↓          ↘ (se há não conformidades)
  │      [Agente 4: Recomendações LLM]  ← Claude sugere correcções
  ↓          ↙
[Agente 5: Sintetizador]     ← gera .docx + .xlsx + .json
  ↓
END
```

---

## Parte 0 — Setup do Ambiente

In [ ]:
# ============================================================
# Instalação das bibliotecas necessárias (Google Colab)
# ============================================================
!pip install langgraph langchain langchain-anthropic anthropic ifcopenshell pandas openpyxl python-docx -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 478.8/478.8 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 24.3 MB/s eta 0:00:00


In [ ]:
# ============================================================
# Configuração da API Key (Anthropic / Claude)
#
# JUSTIFICAÇÃO: É necessário utilizar uma chave para o Agente de Recomendações
#               poder chamar o Claude (LLM). As outras partes do sistema não
#               precisam de um LLM, apenas de lógica Python.
# ============================================================

import os

# Carregamento a partir do separador "Secrets" do Google Colab
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
print("✅ API Key carregada via Colab Secrets")

✅ API Key carregada via Colab Secrets


In [ ]:
# ============================================================
# Upload do ficheiro IFC
#
# JUSTIFICAÇÃO: O Google Colab não tem acesso ao disco local.
#               É necessário fazer o upload do ficheiro .ifc para
#               o ambiente Colab.
# ============================================================

IFC_PATH = "edificio_agricola_com_erro.ifc"
print(f"📁 Caminho do ficheiro IFC: {IFC_PATH}")

📁 Caminho do ficheiro IFC: edificio_agricola_com_erro.ifc


In [ ]:
# ============================================================
# Imports principais
#
# JUSTIFICAÇÃO para as principais bibliotecas:
#   - TypedDict:    define o "contrato" de dados entre os agentes
#   - Annotated:    necessário para o add_messages (acumulador de mensagens)
#   - Optional:     permite campos que começam a "None" (ainda por preencher)
#   - StateGraph:   o construtor do grafo LangGraph
#   - START, END:   marcadores especiais do grafo
#   - add_messages: acumula mensagens em vez de as substituir
#   - HumanMessage: mensagem do utilizador enviada ao LLM
#   - AIMessage:    mensagem registada por cada agente no histórico
#   - SystemMessage: instrução de papel/contexto enviada ao LLM
#   - ChatAnthropic: o LLM que usamos no agente de recomendações e sintetizador
#   - DocxDocument: criação do relatório Word (.docx)
#   - openpyxl:     criação da checklist Excel (.xlsx)
# ============================================================

import json
import re
from datetime import datetime
from typing import TypedDict, Annotated, List, Optional

# LangGraph — os blocos de construção do sistema multi-agente
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# LangChain — interface com o LLM e sistema de mensagens
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Saídas Word e Excel
from docx import Document as DocxDocument
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# LLM — modelo Claude (usado no Agente de Recomendações e no Sintetizador)
llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0, max_tokens=1024)

print("✅ Imports e LLM carregados com sucesso e prontos para a acção!")

✅ Imports e LLM carregados com sucesso e prontos para a acção!


---

## Parte 1 — O Estado Partilhado

### Conceito: O Estado como "pasta de projecto"

No LangGraph, o **State** (Estado) é o dicionário partilhado que todos os agentes lêem e preenchem.
Funciona como uma pasta de projecto de uma equipa real, por exemplo:
- O topógrafo preenche os dados relativos à implantação
- O arquitecto preenche os dados sobre áreas e programa
- O engenheiro preenche os cálculos estruturais

Cada especialista **só preenche a sua secção** — Faz o preenchimento e regista uma mensagem no histórico partilhado. O Sintetizador lê esse histórico completo e utiliza-o como base para gerar o relatório final.


### Regras do TypedDict

- Cada campo tem um tipo declarado (`str`, `dict`, `list`, etc.)
- `Optional[X]` significa que o campo pode estar `None` (antes de ser preenchido)
- Um nó devolve **apenas as chaves que alterou** — o LangGraph faz o merge automaticamente

In [ ]:
# ============================================================
# Estado do Sistema Multi-Agente
#
# Cada campo é da responsabilidade de um agente.
# O campo 'messages' acumula as mensagens de todos os agentes ao longo da
# execução, formando o histórico completo que o Sintetizador usa para gerar
# o relatório final.
# ============================================================

class EstadoInspectorAgricola(TypedDict):
    """Estado partilhado entre todos os agentes do sistema."""
    # Histórico de mensagens
    messages: Annotated[list, add_messages]

    # Caminho para o modelo IFC dado pelo usuário (criado para incluir este passo no Estado)
    caminho_ifc: str

    # Agente 1: Extrator de dados do modelo IFC
    elementos_ifc: Optional[dict]

    # Agente 2: Verificador de conformidades
    verificacao: Optional[dict]

    # Agente 3: Quantificador dos dados de construção do modelo IFC
    medicoes: Optional[dict]

    # Agente 4: Promotor de recomendações (condicional)
    recomendacoes_llm: Optional[str]

    # Agente 5: Outputs criados pelo Sintetizador
    saidas: Optional[dict]

print("✅ EstadoInspectorAgricola definido com todos os campos para roteamento entre agentes (routing).")

✅ EstadoInspectorAgricola definido com todos os campos para roteamento entre agentes (routing).


---

## Parte 2 — Os Agentes (Nós do Grafo)

### Conceito: Um Nó = Uma Função Python

Um **nó** no LangGraph é simplesmente uma função que:
1. Recebe o estado actual (dicionário completo)
2. Faz o seu trabalho (lógica, IfcOpenShell, LLM, etc.)
3. Devolve **apenas as chaves que alterou**

O LangGraph gere a passagem de estado entre os nós.

---

### Agente 1 — Extrator IFC

**Responsabilidade:** Abrir o ficheiro `.ifc` com IfcOpenShell e extrair os elementos relevantes para um dicionário Python limpo.

**Porquê separar este agente?** Porque a leitura do IFC é um processo simples de input/output.
Ao separar, podemos:
- Testar a extração independentemente
- Substituir por outro parser sem tocar nos agentes seguintes
- Ver claramente o que entra vs. o que o verificador usa

In [ ]:
# ============================================================
# AGENTE 1 — Extrator IFC
#
# Usa IfcOpenShell para ler o ficheiro .ifc e extrair:
#   - Nome do projecto e pisos
#   - Paredes (tipos e contagens)
#   - Janelas (nome, altura, largura)
#   - Portas  (nome, altura, largura)
#   - Lajes   (contagem e área de pavimentos interiores)
#   - Espaços (número, nome funcional)
#
# NOTA: As lajes são filtradas por PredefinedType para calcular
#       apenas a área de pavimentos interiores (FLOOR), excluindo
#       coberturas (ROOF) e lajes de terreno (BASESLAB).
# ============================================================

def agente_extrator_ifc(estado: EstadoInspectorAgricola) -> dict:
    """Lê o ficheiro IFC e extrai todos os elementos relevantes."""
    import ifcopenshell
    import ifcopenshell.util.element

    caminho = estado["caminho_ifc"]
    print(f"\n🔍 [Agente Extrator IFC] A abrir: {caminho}")

    modelo = ifcopenshell.open(caminho)

    # Projecto
    projecto_ifc = modelo.by_type("IfcProject")
    nome_projecto = projecto_ifc[0].LongName or projecto_ifc[0].Name if projecto_ifc else "Projecto IFC"
    # Limpa caracteres IFC especiais
    nome_projecto = re.sub(r'\\X\\([0-9A-F]{2})', lambda m: chr(int(m.group(1), 16)), nome_projecto)

    # Pisos
    storeys_raw = modelo.by_type("IfcBuildingStorey")
    pisos = [
        {"nome": s.Name or "Sem nome", "elevacao": round(s.Elevation or 0.0, 3)}
        for s in storeys_raw
    ]
    print(f"   ✅ Projecto: {nome_projecto}")
    print(f"   ✅ Pisos: {len(pisos)} ({', '.join(p['nome'] for p in pisos)})")

    # Paredes
    paredes_std  = modelo.by_type("IfcWallStandardCase")
    paredes_gen  = modelo.by_type("IfcWall")
    todas_paredes = list(paredes_std) + list(paredes_gen)

    # Agrupar por tipo (nome do tipo Revit contém a espessura)
    tipos_parede = {}
    for p in todas_paredes:
        tipo = p.Name or "Desconhecido"
        # Extrair só o nome sem o ID numérico REVIT (ex: "Basic Wall:Parede Exterior 370 mm")
        tipo_limpo = re.sub(r':\d+$', '', tipo)
        tipos_parede[tipo_limpo] = tipos_parede.get(tipo_limpo, 0) + 1
    print(f"   ✅ Paredes: {len(todas_paredes)} ({len(tipos_parede)} tipos)")

    # Janelas
    janelas_raw = modelo.by_type("IfcWindow")
    janelas = []
    for j in janelas_raw:
        janelas.append({
            "nome": re.sub(r':\d+$', '', j.Name or "Janela"),
            "altura_m": round(j.OverallHeight or 0.0, 3),
            "largura_m": round(j.OverallWidth or 0.0, 3),
        })
    print(f"   ✅ Janelas: {len(janelas)}")

    # Portas
    portas_raw = modelo.by_type("IfcDoor")
    portas = []
    for p in portas_raw:
        portas.append({
            "nome": re.sub(r':\d+$', '', p.Name or "Porta"),
            "altura_m": round(p.OverallHeight or 0.0, 3),
            "largura_m": round(p.OverallWidth or 0.0, 3),
        })
    print(f"   ✅ Portas: {len(portas)}")

    # Lajes — filtrar por tipo para calcular apenas pavimentos interiores
    # FLOOR = pavimento interior | ROOF = cobertura | BASESLAB = laje de terreno
    lajes_raw = modelo.by_type("IfcSlab")
    area_piso_m2 = 0.0
    lajes_piso = []
    for laje in lajes_raw:
        tipo_laje = getattr(laje, 'PredefinedType', None)
        if tipo_laje not in ('ROOF', 'BASESLAB'):
            lajes_piso.append(laje)
            try:
                psets = ifcopenshell.util.element.get_psets(laje)
                area = (
                    psets.get("Qto_SlabBaseQuantities", {}).get("NetArea")
                    or psets.get("Qto_SlabBaseQuantities", {}).get("GrossArea")
                    or psets.get("BaseQuantities", {}).get("NetArea")
                )
                if area:
                    area_piso_m2 += area
            except Exception:
                pass
    print(f"   ✅ Lajes: {len(lajes_raw)} total | {len(lajes_piso)} pavimentos interiores | {area_piso_m2:.2f} m²")

    # Espaços
    espacos_raw = modelo.by_type("IfcSpace")
    espacos = []
    for e in espacos_raw:
        nome = e.LongName or e.Name or "Sem nome"
        # Limpa caracteres especiais IFC
        nome = re.sub(r'\\X\\([0-9A-F]{2})', lambda m: chr(int(m.group(1), 16)), nome)
        numero = e.Name or "-"

        # Tenta obter área do pset (pode não estar disponível no Revit IFC2X3)
        area = None
        try:
            psets = ifcopenshell.util.element.get_psets(e)
            area = (
                psets.get("Qto_SpaceBaseQuantities", {}).get("NetFloorArea")
                or psets.get("BaseQuantities", {}).get("NetFloorArea")
                or psets.get("Pset_SpaceCommon", {}).get("NetPlannedArea")
            )
        except Exception:
            area = None

        espacos.append({"numero": numero, "nome": nome, "area_m2": area})

    print(f"   ✅ Espaços: {len(espacos)}")

    # Resultado
    elementos = {
        "projecto":    nome_projecto,
        "pisos":       pisos,
        "paredes":     {"total": len(todas_paredes), "tipos": tipos_parede},
        "janelas":     janelas,
        "portas":      portas,
        "lajes":       {"total": len(lajes_raw)},
        "area_piso_m2": round(area_piso_m2, 2),
        "espacos":     espacos,
    }

    # Este nó devolve APENAS as chaves que alterou
    # Registamos também uma mensagem no histórico partilhado
    return {
        "elementos_ifc": elementos,
        "messages": [AIMessage(content=
            f"[Agente Extrator] Modelo: {nome_projecto} | "
            f"{len(todas_paredes)} paredes | {len(janelas)} janelas | "
            f"{len(portas)} portas | {len(lajes_raw)} lajes | {len(espacos)} espaços"
        )]
    }


print("✅ agente_extrator_ifc definido")

✅ agente_extrator_ifc definido


---

### Agente 2 — Verificador de Conformidade

**Responsabilidade:** Analisar os dados extraídos e verificar critérios técnicos específicos para uma edificação agrícola.

**Critérios verificados:**

| Critério | Referência | Limite |
|---|---|---|
| Rácio de ventilação natural | Portaria 702/80 | Área janelas ≥ 1/10 da área de pavimento |
| Largura mínima das portas | DL 163/2006 | ≥ 0.80m |
| Espaços técnicos identificados | Programa do edifício | Oficina, Parque Máquinas, Armazém Fitofármacos |
| Armazém de fitofármacos | DL 173/2005 | Deve ser um espaço separado e fechado |
| Instalações sanitárias | Decreto-Lei 347/93 | IS Masc. e Fem. separadas |

In [ ]:
# ============================================================
# AGENTE 2 — Verificador de Conformidade
#
# Verifica critérios técnicos para edificações agrícolas
# com base em legislação portuguesa:
#
#   Portaria 702/80       — ventilação natural em edifícios agrícolas
#   Portaria 987/93       — condições de higiene e segurança no trabalho
#   DL 347/93             — segurança, higiene e saúde no trabalho
#   DL 173/2005           — armazenamento de produtos fitofarmacêuticos
#   DL 163/2006           — acessibilidade (largura mínima de portas)
#
# Não usa LLM — é lógica Python pura (mais rápido, auditável,
# determinístico: mesmos dados → mesmo resultado sempre).
# ============================================================

def agente_verificador(estado: EstadoInspectorAgricola) -> dict:
    """Verifica conformidade técnica dos elementos do modelo IFC."""
    print("\n📐 [Agente Verificador] A verificar critérios técnicos...")

    dados = estado["elementos_ifc"]
    aprovacoes = []
    nao_conformidades = []

    # CRITÉRIO 1: Largura mínima das portas
    # DL 163/2006 (Portugal) — Regime da acessibilidade:
    # largura livre mínima de 0.80m em edifícios de utilização colectiva
    portas_estreitas = [p for p in dados["portas"] if p["largura_m"] < 0.80]

    if portas_estreitas:
        nao_conformidades.append(
            f"Acessibilidade (DL 163/2006) — {len(portas_estreitas)} porta(s) com largura < 0.80m: "
            + ", ".join(f"{p['nome']} ({p['largura_m']:.2f}m)" for p in portas_estreitas)
        )
    else:
        aprovacoes.append(
            f"Acessibilidade (DL 163/2006) — Todas as {len(dados['portas'])} portas têm largura ≥ 0.80m "
            f"(mín. encontrado: {min(p['largura_m'] for p in dados['portas']):.2f}m)"
        )

    # CRITÉRIO 2: Ventilação natural
    # Portaria 702/80 — edifícios agrícolas devem ter ventilação natural
    # adequada. A área de vãos de ventilação deve ser no mínimo 1/10
    # da área do pavimento (10%).
    # O Agente Extrator calcula a área dos pavimentos interiores excluindo
    # coberturas e lajes de terreno através do atributo PredefinedType.
    area_janelas = sum(j["largura_m"] * j["altura_m"] for j in dados["janelas"])
    area_piso = dados.get("area_piso_m2", 0)

    if area_janelas == 0:
        nao_conformidades.append(
            "Ventilação (Portaria 702/80) — Nenhuma janela encontrada no modelo"
        )
    elif area_piso > 0:
        racio = area_janelas / area_piso
        minimo = 0.10
        if racio >= minimo:
            aprovacoes.append(
                f"Ventilação (Portaria 702/80) — Rácio de {racio:.1%} "
                f"(janelas: {area_janelas:.2f} m² / piso: {area_piso:.2f} m²). "
                f"Cumpre o mínimo de {minimo:.0%}."
            )
        else:
            nao_conformidades.append(
                f"Ventilação (Portaria 702/80) — Rácio de {racio:.1%} "
                f"(janelas: {area_janelas:.2f} m² / piso: {area_piso:.2f} m²). "
                f"Não cumpre o mínimo de {minimo:.0%}."
            )
    else:
        aprovacoes.append(
            f"Ventilação (Portaria 702/80) — Área de janelas: {area_janelas:.2f} m² "
            f"({len(dados['janelas'])} janelas). "
            f"Área de pavimento não exportada — verificar rácio em obra."
        )

    # CRITÉRIO 3: Espaços funcionais obrigatórios
    # Programa funcional mínimo para edificação de apoio agrícola
    # conforme boas práticas e regulamentação sectorial (Portugal)
    nomes_espacos = [e["nome"].lower() for e in dados["espacos"]]

    espacos_necessarios = {
        "oficina ou ferramentaria": ["oficina", "ferramentaria"],
        "armazém de fitofármacos":  ["fitof", "armazém", "armaz"],
        "parque de máquinas":       ["parque", "máquinas", "maquinas"],
        "instalações sanitárias":   ["i.s.", "instalação", "sanitár", "wc"],
    }

    for descricao, palavras_chave in espacos_necessarios.items():
        encontrado = any(
            any(kw in nome for kw in palavras_chave)
            for nome in nomes_espacos
        )
        if encontrado:
            aprovacoes.append(f"Programa funcional — Espaço '{descricao}' identificado no modelo")
        else:
            nao_conformidades.append(f"Programa funcional — Espaço '{descricao}' não encontrado no modelo")

    # CRITÉRIO 4: Armazém de fitofármacos separado
    # DL 173/2005 — os produtos fitofarmacêuticos devem ser
    # armazenados em local separado, ventilado, de acesso condicionado
    # e com sinalética obrigatória.
    fitof_espacos = [e for e in dados["espacos"] if "fitof" in e["nome"].lower()]
    if fitof_espacos:
        nome_fitof = fitof_espacos[0]["nome"]
        aprovacoes.append(
            f"DL 173/2005 — '{nome_fitof}' existe como espaço autónomo. "
            f"Verificar em obra: ventilação directa para o exterior e sinalética obrigatória."
        )

    # CRITÉRIO 5: IS separadas por género
    # DL 347/93 / Portaria 987/93 — em locais de trabalho com
    # trabalhadores de ambos os géneros, as instalações sanitárias
    # devem ser separadas e devidamente identificadas.
    tem_is_fem  = any("feminina" in n or "duche f" in n or "cabine f" in n for n in nomes_espacos)
    tem_is_masc = any("masculina" in n or "duche m" in n or "cabine m" in n for n in nomes_espacos)

    if tem_is_fem and tem_is_masc:
        aprovacoes.append(
            "Higiene (DL 347/93 / Portaria 987/93) — IS Feminina e IS Masculina presentes e separadas"
        )
    elif tem_is_fem or tem_is_masc:
        nao_conformidades.append(
            "Higiene (DL 347/93 / Portaria 987/93) — Apenas um género de IS identificado; verificar separação"
        )
    else:
        nao_conformidades.append(
            "Higiene (DL 347/93 / Portaria 987/93) — IS não identificadas no modelo"
        )

    # Resumo
    total  = len(aprovacoes) + len(nao_conformidades)
    pct    = round(len(aprovacoes) / total * 100) if total > 0 else 0
    resumo = f"{len(aprovacoes)} / {total} critérios aprovados ({pct}%)"

    print(f"   ✅ Aprovações: {len(aprovacoes)} | ❌ Não Conformidades: {len(nao_conformidades)}")
    print(f"   Conformidade geral: {resumo}")

    return {
        "verificacao": {
            "aprovacoes": aprovacoes,
            "nao_conformidades": nao_conformidades,
            "resumo": resumo,
        },
        "messages": [AIMessage(content=
            f"[Agente Verificador] {len(aprovacoes)} aprovações | "
            f"{len(nao_conformidades)} não conformidades | "
            f"Conformidade: {resumo}"
        )]
    }


print("✅ agente_verificador definido")

✅ agente_verificador definido


---

### Agente 3 — Quantificador

**Responsabilidade:** Calcular métricas e medições dos elementos do modelo.

**Diferente do Verificador** por terem responsabilidades distintas:
- O Verificador responde à pergunta *"está conforme?"*
- O Quantificador responde à pergunta *"quantos há?"*

Em projectos reais, o quantificador alimenta orçamentos — pode ser um agente autónomo com acesso a bases de preços (SINAPI, BDL, etc.).

In [ ]:
# ============================================================
# AGENTE 3 — Quantificador
#
# Calcula métricas e medições dos elementos do modelo:
#   - Área total de janelas (m²)
#   - Contagens por tipo de elemento
#   - Distribuição de tipos de parede
#   - Rácio janelas/pavimentos (indicador de iluminação)
# ============================================================

def agente_quantificador(estado: EstadoInspectorAgricola) -> dict:
    """Calcula métricas e medições dos elementos IFC."""
    print("\n📊 [Agente Quantificador] A calcular medições...")

    dados = estado["elementos_ifc"]

    # Área de janelas: largura × altura de cada janela
    area_janelas = sum(j["largura_m"] * j["altura_m"] for j in dados["janelas"])

    # Contagem de janelas por tipo
    tipos_janela = {}
    for j in dados["janelas"]:
        tipos_janela[j["nome"]] = tipos_janela.get(j["nome"], 0) + 1

    # Contagem de portas por tipo
    tipos_porta = {}
    for p in dados["portas"]:
        tipos_porta[p["nome"]] = tipos_porta.get(p["nome"], 0) + 1

    # Espessuras de paredes (extraídas do nome do tipo Revit)
    espessuras = sorted(set(
        int(m) for tipo in dados["paredes"]["tipos"].keys()
        for m in re.findall(r'(\d+)\s*mm', tipo)
    ))

    medicoes = {
        "n_janelas":       len(dados["janelas"]),
        "area_janelas_m2": round(area_janelas, 2),
        "tipos_janela":    tipos_janela,
        "n_portas":        len(dados["portas"]),
        "tipos_porta":     tipos_porta,
        "n_paredes":       dados["paredes"]["total"],
        "tipos_parede":    dados["paredes"]["tipos"],
        "espessuras_mm":   espessuras,
        "n_lajes":         dados["lajes"]["total"],
        "n_espacos":       len(dados["espacos"]),
        "n_pisos":         len(dados["pisos"]),
    }

    print(f"   ✅ Janelas: {medicoes['n_janelas']} unid. | {medicoes['area_janelas_m2']} m²")
    print(f"   ✅ Portas:  {medicoes['n_portas']} unid.")
    print(f"   ✅ Paredes: {medicoes['n_paredes']} unid. | espessuras: {espessuras} mm")
    print(f"   ✅ Lajes:   {medicoes['n_lajes']} unid.")
    print(f"   ✅ Espaços: {medicoes['n_espacos']} espaços em {medicoes['n_pisos']} pisos")

    return {
        "medicoes": medicoes,
        "messages": [AIMessage(content=
            f"[Agente Quantificador] {medicoes['n_janelas']} janelas "
            f"({medicoes['area_janelas_m2']} m²) | "
            f"{medicoes['n_portas']} portas | "
            f"{medicoes['n_paredes']} paredes"
        )]
    }


print("✅ agente_quantificador definido")

✅ agente_quantificador definido


---

### Agente 4 — Recomendações LLM (Condicional)

**Responsabilidade:** Gerar recomendações técnicas detalhadas para cada não conformidade encontrada pelo Verificador.

**Este é o único agente que usa o LLM (Claude).**

**É condicional** porque:
- Se não houver não conformidades, não há nada para recomendar — poupa tokens e tempo
- A decisão de accionar ou não este agente é feita pela **função de roteamento** que lê o estado

**Porquê não usar o LLM para tudo?** O LLM é não-determinístico e tem custo. Para verificações de conformidade ("é ≥ 0.80m?"), Python puro é mais fiável, mais rápido e auditável.

In [ ]:
# ============================================================
# AGENTE 4 — Recomendações LLM
#
# Este agente só é executado SE houver não conformidades no estado.
# A decisão de o accionar é tomada pela função de roteamento
# rotear_apos_quantificador() definida mais abaixo.
#
# PROMPT DESIGN:
#   - System prompt define o papel e o contexto (edifício agrícola PT)
#   - Human message lista as não conformidades concretas
#   - Pedimos resposta estruturada por não conformidade
# ============================================================

def agente_recomendacoes(estado: EstadoInspectorAgricola) -> dict:
    """Usa o LLM para gerar recomendações técnicas sobre as não conformidades encontradas."""
    print("\n💡 [Agente Recomendações LLM] A gerar recomendações...")

    nao_conformidades = estado["verificacao"]["nao_conformidades"]
    dados = estado["elementos_ifc"]

    # Construção do prompt
    system_prompt = """És um engenheiro técnico especializado em edificações agrícolas e industriais em Portugal.
Para cada não conformidade identificada num modelo BIM (IFC), fornece:
1. Acção correctiva concreta e exequível
2. Referência normativa ou regulamentar aplicável (legislação portuguesa)
3. Prazo estimado de resolução
Sê técnico, directo e objectivo. Usa linguagem de relatório técnico em Português de Portugal."""

    nao_conformidades_texto = "\n".join(f"- {p}" for p in nao_conformidades)
    human_prompt = f"""Projecto: {dados['projecto']}
Tipo: Edificação de apoio agrícola (Casa de Máquinas)

Não Conformidades identificadas no modelo BIM:
{nao_conformidades_texto}

Para cada não conformidade, fornece a acção correctiva, referência normativa e prazo."""

    # Chamada ao LLM
    resposta = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=human_prompt)
    ])

    print("   ✅ Recomendações geradas!")
    return {
        "recomendacoes_llm": resposta.content,
        "messages": [AIMessage(content=
            f"[Agente Recomendações] {resposta.content}"
        )]
    }


print("✅ agente_recomendacoes definido")

✅ agente_recomendacoes definido


---

### Agente 5 — Sintetizador

**Responsabilidade:** Usar o histórico de mensagens de todos os agentes para pedir ao LLM que redija o relatório técnico final, e produzir os três entregáveis: relatório Word (`.docx`), checklist Excel (`.xlsx`) e log JSON (`.json`).

In [ ]:
# ============================================================
# AGENTE 5 — Sintetizador
#
# Usa o histórico de mensagens de todos os agentes anteriores
# para pedir ao LLM que redija o relatório técnico final.
#
# Desta forma, seguimos o padrão da aula 03 — o LLM recebe o contexto
# completo de toda a execução e produz um relatório coerente.
# ============================================================


def gerar_docx(relatorio_txt, dados, verificacao):
    """Converte o relatório de texto para formato Word (.docx)."""
    doc = DocxDocument()

    titulo = doc.add_heading(dados["projecto"], level=0)
    titulo.alignment = WD_ALIGN_PARAGRAPH.CENTER
    sub = doc.add_paragraph(f"Relatório de Inspecção BIM  |  {datetime.now().strftime('%d/%m/%Y')}")
    sub.alignment = WD_ALIGN_PARAGRAPH.CENTER
    doc.add_paragraph()

    for linha in relatorio_txt.split("\n"):
        linha_strip = linha.strip()
        if not linha_strip:
            continue
        if linha_strip.startswith("## "):
            doc.add_heading(linha_strip[3:], level=1)
        elif linha_strip.startswith("### "):
            doc.add_heading(linha_strip[4:], level=2)
        elif linha_strip.startswith("- ") or linha_strip.startswith("* "):
            doc.add_paragraph(linha_strip[2:], style="List Bullet")
        elif linha_strip.startswith("**") and linha_strip.endswith("**"):
            p = doc.add_paragraph()
            p.add_run(linha_strip.strip("*")).bold = True
        else:
            doc.add_paragraph(linha_strip)

    doc.save("relatorio_inspector.docx")


def gerar_xlsx(verificacao, medicoes, dados):
    """Gera a checklist de conformidade em formato Excel (.xlsx)."""
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Conformidade"
    ws.sheet_view.showGridLines = False

    cor_titulo = PatternFill("solid", fgColor="1B5E20")
    cor_ok = PatternFill("solid", fgColor="E8F5E9")
    cor_nok = PatternFill("solid", fgColor="FFEBEE")
    cor_hdr = PatternFill("solid", fgColor="2E7D32")
    fonte_titulo = Font(bold=True, color="FFFFFF", name="Calibri", size=12)
    fonte_hdr = Font(bold=True, color="FFFFFF", name="Calibri", size=11)
    al_centro = Alignment(horizontal="center", vertical="center", wrap_text=True)
    al_esq = Alignment(horizontal="left", vertical="center", wrap_text=True)
    borda = Border(
        left=Side(style="thin", color="BDBDBD"),
        right=Side(style="thin", color="BDBDBD"),
        top=Side(style="thin", color="BDBDBD"),
        bottom=Side(style="thin", color="BDBDBD")
    )

    ws.column_dimensions["A"].width = 12
    ws.column_dimensions["B"].width = 65

    ws.merge_cells("A1:B1")
    c = ws["A1"]
    c.value = f"CHECKLIST DE CONFORMIDADE — {dados['projecto'].upper()}"
    c.fill = cor_titulo
    c.font = fonte_titulo
    c.alignment = al_centro
    ws.row_dimensions[1].height = 30

    ws.merge_cells("A2:B2")
    c = ws["A2"]
    c.value = f"Inspector BIM  |  {datetime.now().strftime('%d/%m/%Y')}"
    c.fill = PatternFill("solid", fgColor="388E3C")
    c.font = Font(color="FFFFFF", name="Calibri", size=10, italic=True)
    c.alignment = al_centro
    ws.row_dimensions[2].height = 18

    for col, hdr in zip(["A", "B"], ["Estado", "Critério / Descrição"]):
        c = ws[f"{col}3"]
        c.value = hdr
        c.fill = cor_hdr
        c.font = fonte_hdr
        c.alignment = al_centro
        c.border = borda
    ws.row_dimensions[3].height = 22

    linha = 4
    for a in verificacao["aprovacoes"]:
        ws.cell(linha, 1, "OK").fill = cor_ok
        ws.cell(linha, 1).font = Font(name="Calibri", size=10, color="1B5E20", bold=True)
        ws.cell(linha, 1).alignment = al_centro
        ws.cell(linha, 1).border = borda
        ws.cell(linha, 2, a).fill = cor_ok
        ws.cell(linha, 2).font = Font(name="Calibri", size=10, color="1B5E20")
        ws.cell(linha, 2).alignment = al_esq
        ws.cell(linha, 2).border = borda
        ws.row_dimensions[linha].height = 28
        linha += 1

    for n in verificacao["nao_conformidades"]:
        ws.cell(linha, 1, "NOK").fill = cor_nok
        ws.cell(linha, 1).font = Font(name="Calibri", size=10, color="B71C1C", bold=True)
        ws.cell(linha, 1).alignment = al_centro
        ws.cell(linha, 1).border = borda
        ws.cell(linha, 2, n).fill = cor_nok
        ws.cell(linha, 2).font = Font(name="Calibri", size=10, color="B71C1C")
        ws.cell(linha, 2).alignment = al_esq
        ws.cell(linha, 2).border = borda
        ws.row_dimensions[linha].height = 32
        linha += 1

    ws.merge_cells(f"A{linha}:B{linha}")
    c = ws.cell(linha, 1, f"RESULTADO: {verificacao['resumo']}")
    c.fill = cor_titulo
    c.font = fonte_titulo
    c.alignment = al_centro
    ws.row_dimensions[linha].height = 26

    ws2 = wb.create_sheet("Medicoes")
    ws2.sheet_view.showGridLines = False
    ws2.column_dimensions["A"].width = 35
    ws2.column_dimensions["B"].width = 25

    ws2.merge_cells("A1:B1")
    c = ws2["A1"]
    c.value = "MEDICOES"
    c.fill = cor_titulo
    c.font = fonte_titulo
    c.alignment = al_centro
    ws2.row_dimensions[1].height = 26

    for col, hdr in zip(["A", "B"], ["Elemento", "Quantidade"]):
        c = ws2[f"{col}2"]
        c.value = hdr
        c.fill = cor_hdr
        c.font = fonte_hdr
        c.alignment = al_centro
        c.border = borda
    ws2.row_dimensions[2].height = 22

    rows = [
        ("Janelas (unidades)", medicoes["n_janelas"]),
        ("Janelas (area m2)", medicoes["area_janelas_m2"]),
        ("Portas (unidades)", medicoes["n_portas"]),
        ("Paredes (total)", medicoes["n_paredes"]),
        ("Espessuras de parede (mm)", str(medicoes["espessuras_mm"])),
        ("Lajes", medicoes["n_lajes"]),
        ("Espacos funcionais", medicoes["n_espacos"]),
        ("Pisos", medicoes["n_pisos"]),
    ]
    for i, (elem, val) in enumerate(rows, 3):
        fill = PatternFill("solid", fgColor="ECEFF1" if i % 2 == 0 else "FFFFFF")
        ws2.cell(i, 1, elem).fill = fill
        ws2.cell(i, 1).alignment = al_esq
        ws2.cell(i, 1).border = borda
        ws2.cell(i, 2, val).fill = fill
        ws2.cell(i, 2).alignment = al_centro
        ws2.cell(i, 2).border = borda
        ws2.row_dimensions[i].height = 20

    wb.save("checklist_conformidade.xlsx")


def gerar_json(estado):
    """Gera o log estruturado em formato JSON."""
    log = {
        "meta": {
            "sistema": "Inspector BIM — Edificação de Apoio Agricola",
            "versao": "1.0",
            "data": datetime.now().isoformat(),
            "ficheiro_ifc": estado["caminho_ifc"],
            "projecto": estado["elementos_ifc"]["projecto"],
        },
        "elementos_extraidos": estado["elementos_ifc"],
        "verificacao": estado["verificacao"],
        "medicoes": estado["medicoes"],
        "recomendacoes": estado.get("recomendacoes_llm"),
    }
    with open("log_inspector.json", "w", encoding="utf-8") as f:
        json.dump(log, f, ensure_ascii=False, indent=2)


def agente_sintetizador(estado: EstadoInspectorAgricola) -> dict:
    """Usa o histórico de mensagens para gerar o relatório final com o LLM."""
    print("\n📄 [Agente Sintetizador] A gerar relatório final...")

    dados = estado["elementos_ifc"]
    verificacao = estado["verificacao"]
    medicoes = estado["medicoes"]

    # O histórico completo de todos os agentes está em estado["messages"]
    # Passamos directamente ao LLM como contexto para gerar o relatório
    mensagens_para_llm = [
        SystemMessage(content=
            "És um engenheiro técnico especializado em edificações agrícolas em Portugal. "
            "Com base nos relatórios dos agentes abaixo, redige um relatório técnico final "
            "em Português de Portugal. Estrutura o relatório com: "
            "1. Sumário Executivo, "
            "2. Dados do Modelo IFC, "
            "3. Verificação de Conformidade, "
            "4. Medições, "
            "5. Recomendações Técnicas (se aplicável), "
            "6. Conclusão. "
            "Usa linguagem técnica e formal."
        )
    ] + estado["messages"]

    resposta = llm.invoke(mensagens_para_llm)
    relatorio = resposta.content

    # Guardar o relatório em .txt
    with open("relatorio_final.txt", "w", encoding="utf-8") as f:
        f.write(relatorio)

    # Gerar os três entregáveis
    gerar_docx(relatorio, dados, verificacao)
    gerar_xlsx(verificacao, medicoes, dados)
    gerar_json(estado)

    print("   ✅ Relatório final gerado pelo LLM!")
    print("   ✅ relatorio_inspector.docx gerado")
    print("   ✅ checklist_conformidade.xlsx gerada")
    print("   ✅ log_inspector.json gerado")

    return {
        "saidas": {
            "relatorio_txt": "relatorio_final.txt",
            "relatorio_docx": "relatorio_inspector.docx",
            "checklist_xlsx": "checklist_conformidade.xlsx",
            "log_json": "log_inspector.json",
        },
        "messages": [AIMessage(content="[Sintetizador] Relatório técnico final gerado.")]
    }


print("✅ agente_sintetizador definido")

✅ agente_sintetizador definido


---

## Parte 3 — Roteamento Condicional

### Conceito: A função de roteamento

Após o Quantificador, o grafo precisa de decidir:
- Há não conformidades → acionar o Agente de Recomendações LLM
- Sem não conformidades → ir directo ao Sintetizador

A **função de roteamento** lê o estado e devolve uma *string* com o nome do próximo nó.
Ela **nunca modifica** o estado — só observa e decide.

In [ ]:
# ============================================================
# Função de Roteamento Condicional
#
# Decide o próximo nó após o Quantificador:
#   - Se houver não conformidades → agente_recomendacoes (LLM)
#   - Se não houver         → agente_sintetizador  (output directo)
# ============================================================

def rotear_apos_quantificador(estado: EstadoInspectorAgricola) -> str:
    """
    Roteamento condicional após o Agente Quantificador.

    Retorna:
        'agente_recomendacoes' se existirem não conformidades
        'agente_sintetizador'  se o modelo estiver conforme
    """
    nao_conformidades = estado["verificacao"]["nao_conformidades"]

    if nao_conformidades:
        print(f"⚠️  [Roteador] {len(nao_conformidades)} não conformidade(s) → a accionar Agente Recomendações")
        return "agente_recomendacoes"
    else:
        print("✅ [Roteador] Sem não conformidades → a ir directo para Sintetizador")
        return "agente_sintetizador"


print("✅ Função de roteamento definida")

✅ Função de roteamento definida


---

## Parte 4 — Construção e Execução do Grafo

### Conceito: Montar o grafo passo a passo

Agora que todos os nós e a função de roteamento estão definidos, montamos o grafo:

1. **`StateGraph(Estado)`** → cria o grafo com o nosso contrato de estado
2. **`add_node(nome, função)`** → regista cada agente como nó
3. **`add_edge(origem, destino)`** → aresta fixa (fluxo sempre igual)
4. **`add_conditional_edges(origem, função, mapa)`** → aresta condicional
5. **`compile()`** → valida e gera o grafo executável
6. **`invoke(estado_inicial)`** → executa o grafo

In [ ]:
# ============================================================
# Construção do grafo multi-agente
# ============================================================

# Criar o builder com o Estado definido
builder = StateGraph(EstadoInspectorAgricola)

# Registar cada agente como nó
#   O primeiro argumento é o NOME do nó (usado nas arestas)
#   O segundo argumento é a FUNÇÃO que implementa o agente
builder.add_node("agente_extrator", agente_extrator_ifc)
builder.add_node("agente_verificador", agente_verificador)
builder.add_node("agente_quantificador", agente_quantificador)
builder.add_node("agente_recomendacoes", agente_recomendacoes)
builder.add_node("agente_sintetizador", agente_sintetizador)

# Arestas fixas — fluxo sempre sequencial no pipeline principal
builder.add_edge(START, "agente_extrator")
builder.add_edge("agente_extrator", "agente_verificador")
builder.add_edge("agente_verificador", "agente_quantificador")

# Aresta condicional após o Quantificador
#    - rotear_apos_quantificador() decide o próximo nó
#    - O dicionário mapeia o retorno da função → nome real do nó
builder.add_conditional_edges(
    "agente_quantificador",
    rotear_apos_quantificador,
    {
        "agente_recomendacoes":  "agente_recomendacoes",
        "agente_sintetizador":   "agente_sintetizador",
    }
)

# O Agente Recomendações leva sempre ao Sintetizador
builder.add_edge("agente_recomendacoes", "agente_sintetizador")
builder.add_edge("agente_sintetizador",  END)

# Compilar — o LangGraph valida a estrutura do grafo
grafo = builder.compile()

print("✅ Grafo multi-agente compilado com sucesso!")
print()
print("Fluxo:")
print("  START")
print("    ↓")
print("  [agente_extrator]       ← IfcOpenShell lê o .ifc")
print("    ↓")
print("  [agente_verificador]    ← verifica conformidade")
print("    ↓")
print("  [agente_quantificador]  ← calcula medições")
print("    ↓ (condicional)")
print("  [agente_recomendacoes]? ← LLM (só se há não conformidades)")
print("    ↓")
print("  [agente_sintetizador]   ← gera .docx + .xlsx + .json")
print("    ↓")
print("  END")

✅ Grafo multi-agente compilado com sucesso!

Fluxo:
  START
    ↓
  [agente_extrator]       ← IfcOpenShell lê o .ifc
    ↓
  [agente_verificador]    ← verifica conformidade
    ↓
  [agente_quantificador]  ← calcula medições
    ↓ (condicional)
  [agente_recomendacoes]? ← LLM (só se há não conformidades)
    ↓
  [agente_sintetizador]   ← gera .docx + .xlsx + .json
    ↓
  END


In [ ]:
# ============================================================
# Execução do Sistema Multi-Agente
#
# O campo 'messages' começa vazio — vai sendo preenchido
# por cada agente ao longo da execução.
# ============================================================

print("🚀 A iniciar sistema multi-agente Inspector BIM...")
print("=" * 60)

resultado = grafo.invoke({
    "messages": [],       # começa vazio — acumula durante a execução
    "caminho_ifc": IFC_PATH,
    "elementos_ifc": None,
    "verificacao": None,
    "medicoes": None,
    "recomendacoes_llm": None,
    "saidas": None,
})

print("=" * 60)
print("\n✅ SISTEMA CONCLUÍDO!")
print(f"Relatório: {resultado['saidas']['relatorio_txt']}")
print("\n HISTÓRICO DE MENSAGENS:")
for msg in resultado["messages"]:
    print(f"{msg.content}")

🚀 A iniciar sistema multi-agente Inspector BIM...

🔍 [Agente Extrator IFC] A abrir: edificio_agricola_com_erro.ifc
   ✅ Projecto: CASA DE MÁQUINAS PANASCAL
   ✅ Pisos: 3 (Terreno, Piso Térreo, Cobertura)
   ✅ Paredes: 90 (12 tipos)
   ✅ Janelas: 9
   ✅ Portas: 9
   ✅ Lajes: 22 total | 18 pavimentos interiores | 0.00 m²
   ✅ Espaços: 19

📐 [Agente Verificador] A verificar critérios técnicos...
   ✅ Aprovações: 7 | ❌ Não Conformidades: 1
   Conformidade geral: 7 / 8 critérios aprovados (88%)

📊 [Agente Quantificador] A calcular medições...
   ✅ Janelas: 9 unid. | 16.56 m²
   ✅ Portas:  9 unid.
   ✅ Paredes: 90 unid. | espessuras: [20, 110, 150, 370, 400] mm
   ✅ Lajes:   22 unid.
   ✅ Espaços: 19 espaços em 3 pisos
⚠️  [Roteador] 1 não conformidade(s) → a accionar Agente Recomendações

💡 [Agente Recomendações LLM] A gerar recomendações...
   ✅ Recomendações geradas!

📄 [Agente Sintetizador] A gerar relatório final...
   ✅ Relatório final gerado pelo LLM!
   ✅ relatorio_inspector.docx ger

In [ ]:
# ============================================================
# Visualizar o relatório final no notebook
# ============================================================

print(resultado["saidas"].get("relatorio_txt", ""))

# Ler e imprimir o conteúdo do ficheiro gerado
with open("relatorio_final.txt", encoding="utf-8") as f:
    print(f.read())

relatorio_final.txt


---

# RELATÓRIO TÉCNICO FINAL
**VERIFICAÇÃO DE CONFORMIDADE REGULAMENTAR**

**Projecto:** CASA DE MÁQUINAS PANASCAL  
**Localização:** [A definir]  
**Data da análise:** [Data actual]  
**Técnico responsável:** [Nome]  
**Cédula profissional:** [Número]

---

## 1. SUMÁRIO EXECUTIVO

A presente análise técnica incidiu sobre o modelo BIM da edificação agrícola denominada "CASA DE MÁQUINAS PANASCAL". O edifício apresenta conformidade regulamentar em 87,5% dos critérios analisados, registando-se uma não conformidade relacionada com as dimensões de vãos de passagem.

**Principais conclusões:**
- Estrutura geral em conformidade com regulamentação aplicável
- Necessária correcção de uma porta interior
- Recomenda-se intervenção antes da conclusão da obra

---

## 2. DADOS DO MODELO IFC

**Elementos estruturais identificados:**
- **Paredes:** 90 unidades
- **Lajes:** 22 unidades  
- **Vãos:** 18 unidades (9 janelas + 9 portas)
- **Espaços:** 19 compartimentos
- **Área t

In [ ]:
# ============================================================
# Download dos ficheiros gerados (Google Colab)
# ============================================================

from google.colab import files
files.download("relatorio_inspector.docx")
files.download("checklist_conformidade.xlsx")
files.download("log_inspector.json")

print("Os ficheiros estão disponíveis para download.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Os ficheiros estão disponíveis para download.
